In [5]:
from pathlib import Path
import pandas as pd

raw_dir = Path("../data/raw")

activity_file = sorted(raw_dir.glob("building_activity_*.csv"))[-1]
approvals_file = sorted(raw_dir.glob("ba_sa2_*.csv"))[-1]

df_activity = pd.read_csv(activity_file)
df_approvals = pd.read_csv(approvals_file)

print(activity_file.name)
print(approvals_file.name)

building_activity_20261004T125432659303Z.csv
ba_sa2_20261004T125435181638Z.csv


Building Activity Exploration

In [ ]:
# Check 5 first rows
df_activity.head()

,DATAFLOW,MEASURE,REGION,PRICE_ADJ,BLD_WORK_TYPE,SECTOR_OWN,TYPE_BLDG,TSEST,FREQ,TIME_PERIOD,OBS_VALUE,UNIT_MEASURE,UNIT_MULT,OBS_STATUS,OBS_COMMENT
0,ABS:BUILDING_ACTIVITY(1.0.0),M1,2,CVM,TOT,9,TOT,20,Q,2023-Q2,12673855.0,AUD,3,NaN,NaN
1,ABS:BUILDING_ACTIVITY(1.0.0),M1,2,CVM,TOT,9,TOT,20,Q,2023-Q3,12508261.0,AUD,3,NaN,NaN
2,ABS:BUILDING_ACTIVITY(1.0.0),M1,2,CVM,TOT,9,TOT,20,Q,2023-Q4,12660591.0,AUD,3,NaN,NaN
3,ABS:BUILDING_ACTIVITY(1.0.0),M8,AUS,CUR,TOT,9,TOT,10,Q,2023-Q2,240165.0,NUM,0,NaN,NaN
4,ABS:BUILDING_ACTIVITY(1.0.0),M8,AUS,CUR,TOT,9,TOT,10,Q,2023-Q3,234436.0,NUM,0,NaN,NaN


In [9]:
# Check number of fields and records
df_activity.shape

(552, 15)

In [ ]:
# Check column name, data types
df_activity.info()

<class 'pandas.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   DATAFLOW       552 non-null    str    
 1   MEASURE        552 non-null    str    
 2   REGION         552 non-null    str    
 3   PRICE_ADJ      552 non-null    str    
 4   BLD_WORK_TYPE  552 non-null    str    
 5   SECTOR_OWN     552 non-null    int64  
 6   TYPE_BLDG      552 non-null    str    
 7   TSEST          552 non-null    int64  
 8   FREQ           552 non-null    str    
 9   TIME_PERIOD    552 non-null    str    
 10  OBS_VALUE      504 non-null    float64
 11  UNIT_MEASURE   552 non-null    str    
 12  UNIT_MULT      552 non-null    int64  
 13  OBS_STATUS     48 non-null     str    
 14  OBS_COMMENT    0 non-null      float64
dtypes: float64(2), int64(3), str(10)
memory usage: 64.8 KB


In [11]:
# Check total of missing values
df_activity.isna().sum()

DATAFLOW           0
MEASURE            0
REGION             0
PRICE_ADJ          0
BLD_WORK_TYPE      0
SECTOR_OWN         0
TYPE_BLDG          0
TSEST              0
FREQ               0
TIME_PERIOD        0
OBS_VALUE         48
UNIT_MEASURE       0
UNIT_MULT          0
OBS_STATUS       504
OBS_COMMENT      552
dtype: int64

In [12]:
# Check unique values
df_activity.nunique()

DATAFLOW           1
MEASURE            8
REGION             9
PRICE_ADJ          2
BLD_WORK_TYPE      1
SECTOR_OWN         1
TYPE_BLDG          1
TSEST              3
FREQ               1
TIME_PERIOD        3
OBS_VALUE        501
UNIT_MEASURE       2
UNIT_MULT          2
OBS_STATUS         1
OBS_COMMENT        0
dtype: int64

In [15]:
# Check duplicated values
df_activity.duplicated().sum()

np.int64(0)

In [13]:
for column in df.columns:
    print(column, ":", df[column].dropna().unique()[:8])

DATAFLOW : <StringArray>
['ABS:BUILDING_ACTIVITY(1.0.0)']
Length: 1, dtype: str
MEASURE : <StringArray>
['M1', 'M8', 'M3', 'M7', 'M6', 'M2', 'M5', 'M4']
Length: 8, dtype: str
REGION : <StringArray>
['2', 'AUS', '1', '7', '3', '6', '8', '4']
Length: 8, dtype: str
PRICE_ADJ : <StringArray>
['CVM', 'CUR']
Length: 2, dtype: str
BLD_WORK_TYPE : <StringArray>
['TOT']
Length: 1, dtype: str
SECTOR_OWN : [9]
TYPE_BLDG : <StringArray>
['TOT']
Length: 1, dtype: str
TSEST : [20 10 30]
FREQ : <StringArray>
['Q']
Length: 1, dtype: str
TIME_PERIOD : <StringArray>
['2023-Q2', '2023-Q3', '2023-Q4']
Length: 3, dtype: str
OBS_VALUE : [12673855. 12508261. 12660591.   240165.   234436.   224576. 11197317.
 12033408.]
UNIT_MEASURE : <StringArray>
['AUD', 'NUM']
Length: 2, dtype: str
UNIT_MULT : [3 0]
OBS_STATUS : <StringArray>
['m']
Length: 1, dtype: str
OBS_COMMENT : []


In [16]:
# Identify grain
key_columns = [
    "MEASURE",
    "REGION",
    "PRICE_ADJ",
    "BLD_WORK_TYPE",
    "SECTOR_OWN",
    "TYPE_BLDG",
    "TSEST",
    "FREQ",
    "TIME_PERIOD",
]

df_activity.duplicated(subset=key_columns).sum()

np.int64(0)

In [17]:
pd.crosstab(
    df_activity["OBS_VALUE"].isna(),
    df_activity["OBS_STATUS"].fillna("(blank)")
)

OBS_STATUS,(blank),m
OBS_VALUE,,
False,504,0
True,0,48


In [18]:
df_activity.loc[
    df_activity["OBS_VALUE"].isna(),
    ["MEASURE", "REGION", "PRICE_ADJ", "TSEST",
     "TIME_PERIOD", "OBS_STATUS"]
]

,MEASURE,REGION,PRICE_ADJ,TSEST,TIME_PERIOD,OBS_STATUS
18,M3,6,CVM,20,2023-Q2,m
19,M3,6,CVM,20,2023-Q3,m
20,M3,6,CVM,20,2023-Q4,m
24,M3,6,CVM,30,2023-Q2,m
25,M3,6,CVM,30,2023-Q3,m
26,M3,6,CVM,30,2023-Q4,m
33,M3,2,CVM,20,2023-Q2,m
34,M3,2,CVM,20,2023-Q3,m
35,M3,2,CVM,20,2023-Q4,m
54,M3,8,CVM,20,2023-Q2,m
